# <font color="#418FDE" size="6.5" uppercase>**Images Custom**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Represent images as arrays and extract patch-based features. 
- Create graph structures for image neighborhoods and regular grids. 
- Implement simple custom transformers compatible with scikit-learn pipelines. 


## **1. Image Patch Arrays**

### **1.1. Image Array Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_01.jpg?v=1783988147" width="250">



>* Images are numerical grids of pixel values
>* Color images stack red, green, blue channels

>* Pixel positions preserve nearby scene relationships
>* Local patterns support meaningful feature extraction

>* Array shape and channels guide interpretation
>* Consistent scaling improves patch feature quality



In [ ]:
#@title Python Code - Image Array Basics

# This example treats an image as numbers.
# Pixel positions preserve simple spatial relationships.
# A small patch shows local image structure.

import numpy as np
import matplotlib.pyplot as plt

# Build a tiny synthetic grayscale image array.
image = np.array(
    [[20, 20, 20, 220, 220, 220],
     [20, 20, 20, 220, 220, 220],
     [20, 20, 20, 220, 220, 220],
     [60, 60, 60, 180, 180, 180],
     [60, 60, 60, 180, 180, 180],
     [60, 60, 60, 180, 180, 180]],
    dtype=np.uint8,
)

# Validate the expected two-dimensional grayscale shape.
if image.ndim != 2:
    raise ValueError("Expected a two-dimensional grayscale image array.")

# Select one local patch around a brightness boundary.
patch = image[1:4, 2:5]
patch_mean = float(patch.mean())

# Print compact facts about the image and patch.
print(f"Image shape: {image.shape} rows by columns")
print(f"Pixel value range: {image.min()} to {image.max()}")
print(f"Patch shape: {patch.shape} rows by columns")
print(f"Patch mean brightness: {patch_mean:.1f}")
print(f"Patch values: {patch.ravel().tolist()}")

# Display the image and mark the selected patch.
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image, cmap="gray", vmin=0, vmax=255)

# Draw a red rectangle around the selected patch.
rectangle_x = [1.5, 4.5, 4.5, 1.5, 1.5]
rectangle_y = [0.5, 0.5, 3.5, 3.5, 0.5]
ax.plot(rectangle_x, rectangle_y, color="red", linewidth=3, label="3x3 patch")

# Label pixel coordinates for beginner interpretation.
ax.set_title("Synthetic grayscale image array")
ax.set_xlabel("Column index")
ax.set_ylabel("Row index")
ax.legend(loc="lower right")
plt.show()



### **1.2. Extracting Image Patches**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_02.jpg?v=1783988151" width="250">



>* Small image regions capture local visual details
>* Patches become feature vectors for learning

>* Choose dense or sparse patch locations
>* Balance patch size, overlap, and speed

>* Flatten or summarize patches for learning
>* Many patches reveal local repeated patterns



In [ ]:
#@title Python Code - Extracting Image Patches

# This example extracts patches from one synthetic image.
# Each patch is a smaller image array.
# The plot shows where patches were sampled.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.image import extract_patches_2d

# Build a small synthetic grayscale image with simple patterns.
image = np.zeros((12, 12), dtype=float)
image[2:10, 3:5] = 0.7
image[6:8, 1:11] = 1.0

# Extract every possible four by four patch.
patch_size = (4, 4)
patches = extract_patches_2d(image, patch_size)

# Validate the expected patch array shape.
expected_count = (image.shape[0] - 3) * (image.shape[1] - 3)
if patches.shape != (expected_count, 4, 4):
    raise ValueError("Patch extraction produced an unexpected shape.")

# Flatten patches into feature vectors for machine learning.
patch_vectors = patches.reshape(patches.shape[0], -1)
first_patch_mean = round(float(patches[0].mean()), 2)
brightest_index = int(np.argmax(patch_vectors.mean(axis=1)))

print(f"Image shape: {image.shape} pixels")
print(f"Patch array shape: {patches.shape}")
print(f"Flattened feature matrix shape: {patch_vectors.shape}")
print(f"First patch average brightness: {first_patch_mean}")
print(f"Brightest patch index: {brightest_index}")

# Mark a few sampled patch locations on the original image.
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image, cmap="gray", vmin=0, vmax=1)

# Draw rectangles for three example patch positions.
for row, col in [(0, 0), (4, 4), (8, 8)]:
    rectangle = plt.Rectangle((col - 0.5, row - 0.5), 4, 4, fill=False)
    ax.add_patch(rectangle)

ax.set_title("Synthetic image with three 4x4 patches")
ax.set_xlabel("Column index")
ax.set_ylabel("Row index")
plt.show()



### **1.3. Rebuilding Image Patches**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_01_03.jpg?v=1783988149" width="250">



>* Reassemble patches into full image arrays
>* Connect local analysis to global outputs

>* Overlapping patches share pixel locations
>* Average overlaps for smoother reconstruction

>* Match dimensions, patch size, and order
>* Rebuild aligned maps for visual analysis



In [ ]:
#@title Python Code - Rebuilding Image Patches

# Rebuild overlapping patches into one synthetic image.
# Averaging overlap restores shared pixel locations smoothly.
# The plot shows the reconstructed patch image.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.image import extract_patches_2d

# Create a small synthetic grayscale image.
image = np.arange(36, dtype=float).reshape(6, 6)
patch_size = (3, 3)

# Extract every possible overlapping three by three patch.
patches = extract_patches_2d(image, patch_size)
expected_patches = (image.shape[0] - 2) * (image.shape[1] - 2)

# Validate that extraction produced the expected patch count.
if patches.shape[0] != expected_patches:
    raise ValueError("Patch count does not match the image shape.")

# Add each patch back into its original location.
rebuilt_sum = np.zeros_like(image)
rebuilt_count = np.zeros_like(image)
patch_index = 0

for row in range(image.shape[0] - patch_size[0] + 1):
    for col in range(image.shape[1] - patch_size[1] + 1):
        rebuilt_sum[row:row + 3, col:col + 3] += patches[patch_index]
        rebuilt_count[row:row + 3, col:col + 3] += 1
        patch_index += 1

# Average overlapping contributions for each pixel.
if np.any(rebuilt_count == 0):
    raise ValueError("Every pixel must receive at least one patch value.")
rebuilt_image = rebuilt_sum / rebuilt_count

# Print concise checks that connect patches to reconstruction.
print("Original image shape:", image.shape)
print("Patch array shape:", patches.shape)
print("Center pixel overlap count:", int(rebuilt_count[2, 2]))
print("Maximum reconstruction error:", round(float(np.max(np.abs(image - rebuilt_image))), 6))

# Display the reconstructed image as one complete array.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(rebuilt_image, cmap="gray", interpolation="nearest")
ax.set_title("Rebuilt Image From Averaged Patches")
ax.set_xlabel("Column index")
ax.set_ylabel("Row index")
plt.show()



## **2. Image Graphs**

### **2.1. Pixel Neighborhood Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_01.jpg?v=1783988153" width="250">



>* Pixels become nodes connected to neighbors
>* Neighbor relationships reveal meaningful visual patterns

>* Grid position shapes each pixel’s neighbors
>* Weighted edges reveal regions and boundaries

>* Graphs reveal regions, edges, and smoothness
>* Neighbor context supports practical image analysis



In [ ]:
#@title Python Code - Pixel Neighborhood Graphs

# Build a tiny synthetic pixel neighborhood graph.
# Compare four-neighbor and eight-neighbor connections.
# Visualize graph edges over the image grid.

import numpy as np
import matplotlib.pyplot as plt

# A small image keeps every pixel node easy to inspect.
image = np.array(
    [[20, 20, 80, 80], [20, 50, 80, 90], [10, 50, 55, 90]], dtype=np.uint8
)

height, width = image.shape
if height * width != 12:
    raise ValueError("Expected a 3 by 4 teaching image.")

# Convert each row and column position into one node number.
node_numbers = np.arange(height * width).reshape(height, width)

# Neighbor offsets define which nearby pixels receive graph edges.
four_offsets = [(0, 1), (1, 0)]
eight_offsets = [(0, 1), (1, 0), (1, 1), (1, -1)]

# This helper builds undirected edges without leaving the image.
def build_edges(offsets):
    edges = []
    for row in range(height):
        for col in range(width):
            start = int(node_numbers[row, col])
            for row_step, col_step in offsets:
                new_row = row + row_step
                new_col = col + col_step
                if 0 <= new_row < height and 0 <= new_col < width:
                    end = int(node_numbers[new_row, new_col])
                    edges.append((start, end))
    return edges

four_edges = build_edges(four_offsets)
eight_edges = build_edges(eight_offsets)

# Edge weights can describe brightness changes between neighbors.
edge_differences = []
for start, end in four_edges:
    start_value = int(image.flat[start])
    end_value = int(image.flat[end])
    edge_differences.append(abs(start_value - end_value))

print("Synthetic image shape: 3 rows x 4 columns")
print("Pixel nodes:", height * width)
print("Four-neighbor edges:", len(four_edges))
print("Eight-neighbor edges:", len(eight_edges))
print("First four-neighbor edges:", four_edges[:5])
print("First edge brightness differences:", edge_differences[:5])

# Draw the four-neighbor graph on top of the pixel grid.
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(image, cmap="gray", vmin=0, vmax=100)

for start, end in four_edges:
    start_row, start_col = divmod(start, width)
    end_row, end_col = divmod(end, width)
    ax.plot([start_col, end_col], [start_row, end_row], color="cyan", linewidth=2)

for row in range(height):
    for col in range(width):
        label = str(int(node_numbers[row, col]))
        ax.text(col, row, label, color="yellow", ha="center", va="center")

ax.set_title("Four-Neighbor Pixel Graph on a Synthetic Image")
ax.set_xlabel("Column index")
ax.set_ylabel("Row index")
plt.show()



### **2.2. Regular Grid Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_02.jpg?v=1783988157" width="250">



>* Pixels become nodes connected to nearby pixels
>* Grid graphs preserve spatial image context

>* Connect pixels for graph-based image reasoning
>* Predictable grids support core image tasks

>* Choose edge weights to define pixel connections
>* Combine image layout with visual similarity



In [ ]:
#@title Python Code - Regular Grid Graphs

# Build a tiny regular grid graph.
# Connect pixels using four-neighbor adjacency.
# Display nodes, edges, and degrees.

import numpy as np
import matplotlib.pyplot as plt

# A small synthetic image keeps the graph easy to inspect.
image = np.array(
    [[20, 40, 60, 80], [30, 50, 70, 90], [40, 60, 80, 100]],
    dtype=np.uint8,
)

# Validate the image shape before building graph connections.
height, width = image.shape
if height < 2 or width < 2:
    raise ValueError("Use at least a 2 by 2 image for this example.")

# Each pixel becomes one graph node identified by its row and column.
nodes = []
for row in range(height):
    for col in range(width):
        nodes.append((row, col))

# Four-neighbor edges connect right and downward neighbors once each.
edges = []
for row in range(height):
    for col in range(width):
        if col + 1 < width:
            edges.append(((row, col), (row, col + 1)))
        if row + 1 < height:
            edges.append(((row, col), (row + 1, col)))

# Node degree counts how many neighbors each pixel has.
degrees = {}
for node in nodes:
    degrees[node] = 0

for start, end in edges:
    degrees[start] = degrees[start] + 1
    degrees[end] = degrees[end] + 1

print(f"Synthetic image shape: {height} rows by {width} columns")
print(f"Graph nodes: {len(nodes)} pixels")
print(f"Four-neighbor edges: {len(edges)} connections")
print(f"Corner degree: {degrees[(0, 0)]}")
print(f"Interior degree: {degrees[(1, 1)]}")

# Draw the grid graph over the pixel intensity image.
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(image, cmap="gray", origin="upper")

for start, end in edges:
    y_values = [start[0], end[0]]
    x_values = [start[1], end[1]]
    ax.plot(x_values, y_values, color="tab:blue", linewidth=2)

for row, col in nodes:
    ax.scatter(col, row, color="tab:red", s=80, zorder=3)
    ax.text(col, row, str(degrees[(row, col)]), color="white", ha="center", va="center")

ax.set_title("Four-Neighbor Regular Grid Graph")
ax.set_xlabel("Column index")
ax.set_ylabel("Row index")
ax.set_xticks(range(width))
ax.set_yticks(range(height))
plt.show()



### **2.3. Graph Applications**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_02_03.jpg?v=1783988155" width="250">



>* Model image parts and relationships as graphs
>* Segment coherent regions while preserving boundaries

>* Smooth noisy images using graph connections
>* Preserve true edges and important defects

>* Graphs turn image structure into useful features
>* Spatial links improve context-aware image understanding



## **3. Custom Transformers**

### **3.1. Function Based Transforms**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_01.jpg?v=1783988159" width="250">



>* Wrap simple image functions as pipeline steps
>* Apply preprocessing consistently across all data

>* Use functions for fixed image transformations
>* Adapt them for reusable pipeline steps

>* Define predictable image feature transformations
>* Keep pipelines organized, testable, and flexible



In [ ]:
#@title Python Code - Function Based Transforms

# Wrap image functions as pipeline transformers.
# FunctionTransformer gives functions a sklearn interface.
# The pipeline returns compact patch features.

import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load small packaged digit images for a fast example.
digits = load_digits()
images = digits.images.astype(np.float32)
labels = digits.target

# Validate the image shape before building features.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Expected digit images shaped as 8 by 8 arrays.")

# This function converts pixel values to the zero to one range.
def scale_pixels(image_batch):
    return image_batch / 16.0

# This function summarizes four image patches per digit.
def patch_means(image_batch):
    top_left = image_batch[:, :4, :4].mean(axis=(1, 2))
    top_right = image_batch[:, :4, 4:].mean(axis=(1, 2))
    bottom_left = image_batch[:, 4:, :4].mean(axis=(1, 2))
    bottom_right = image_batch[:, 4:, 4:].mean(axis=(1, 2))
    return np.column_stack((top_left, top_right, bottom_left, bottom_right))

# Wrap each plain function as a pipeline-compatible transformer.
feature_pipeline = Pipeline(
    steps=[
        ("scale", FunctionTransformer(scale_pixels, validate=False)),
        ("patch_means", FunctionTransformer(patch_means, validate=False)),
    ]
)

# Split data so the model sees test images only after training.
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)

# Add a simple classifier after the custom feature pipeline.
model = Pipeline(
    steps=[
        ("features", feature_pipeline),
        ("classifier", LogisticRegression(max_iter=300, random_state=42)),
    ]
)

# Fit the full workflow with one standard sklearn call.
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

# Inspect the transformed features for one image.
example_features = feature_pipeline.transform(images[:1])[0]
rounded_features = np.round(example_features, 3)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original image batch shape: {images.shape}")
print(f"Patch feature shape: {feature_pipeline.transform(images).shape}")
print(f"First image patch means: {rounded_features.tolist()}")
print(f"Pipeline test accuracy: {accuracy:.3f}")



### **3.2. TransformerMixin Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_02.jpg?v=1783988160" width="250">



>* Build pipeline-ready image feature transformers
>* Keep preprocessing consistent and reliable

>* Fit stores information; transform converts data
>* TransformerMixin enables smooth pipeline integration

>* Learn only from training data
>* Package transformations for reliable reuse



In [ ]:
#@title Python Code - TransformerMixin Basics

# Build a small custom image transformer.
# TransformerMixin supplies fit_transform pipeline behavior.
# Printed features confirm reusable image preprocessing.

import numpy as np
from sklearn.base import BaseEstimator
from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline

# Tiny synthetic grayscale images keep the example self-contained.
images = np.array(
    [
        [[0, 0, 2, 2], [0, 1, 2, 3], [4, 4, 6, 6], [4, 5, 6, 7]],
        [[7, 6, 5, 4], [6, 5, 4, 3], [3, 2, 1, 0], [2, 1, 0, 0]],
    ],
    dtype=float,
)

# Validate the expected image batch shape.
if images.ndim != 3:
    raise ValueError("Expected images shaped as samples, height, width.")

# This transformer learns one training-set brightness value.
class BrightnessCenterer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.training_mean_ = float(np.mean(X))
        return self

    def transform(self, X):
        return X - self.training_mean_

# This transformer converts each image into four patch averages.
class PatchMeanFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        top_left = X[:, :2, :2].mean(axis=(1, 2))
        top_right = X[:, :2, 2:].mean(axis=(1, 2))
        bottom_left = X[:, 2:, :2].mean(axis=(1, 2))
        bottom_right = X[:, 2:, 2:].mean(axis=(1, 2))
        return np.column_stack((top_left, top_right, bottom_left, bottom_right))

# A pipeline calls fit and transform in the correct order.
feature_pipeline = Pipeline(
    [
        ("center", BrightnessCenterer()),
        ("patch_means", PatchMeanFeatures()),
    ]
)

# TransformerMixin gives the pipeline a convenient fit_transform method.
features = feature_pipeline.fit_transform(images)
rounded_features = np.round(features, 2)

print("Input image batch shape:", images.shape)
print("Learned training mean brightness:", round(feature_pipeline["center"].training_mean_, 2))
print("Output feature matrix shape:", rounded_features.shape)
print("Patch-mean features for image 1:", rounded_features[0].tolist())
print("Patch-mean features for image 2:", rounded_features[1].tolist())



### **3.3. Feature Name Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_B/image_03_03.jpg?v=1783988163" width="250">



>* Preserve meaning of transformed image features
>* Support interpretation, debugging, and real-world trust

>* Use consistent names across pipeline steps
>* Name new features clearly and predictably

>* Consistent names support teamwork and reproducibility
>* Names must match actual transformed features



In [ ]:
#@title Python Code - Feature Name Handling

# Build a transformer with readable feature names.
# Feature names preserve meaning after transformation.
# The output confirms pipeline-compatible naming behavior.

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import sklearn

# This tiny table represents image-derived measurements.
image_features = pd.DataFrame(
    {
        "avg_red": [120, 80, 150, 60],
        "edge_density": [0.30, 0.10, 0.45, 0.05],
        "patch_brightness": [210, 160, 230, 140],
    }
)

# The transformer adds one named summary feature.
class BrightnessRatioAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.array(X.columns, dtype=object)
        return self

    def transform(self, X):
        X_array = np.asarray(X, dtype=float)
        if X_array.shape[1] != len(self.feature_names_in_):
            raise ValueError("Input column count changed after fitting.")
        red_index = list(self.feature_names_in_).index("avg_red")
        bright_index = list(self.feature_names_in_).index("patch_brightness")
        ratio = X_array[:, bright_index] / np.maximum(X_array[:, red_index], 1.0)
        return np.column_stack([X_array, ratio])

    def get_feature_names_out(self, input_features=None):
        base_names = list(self.feature_names_in_)
        return np.array(base_names + ["patch_brightness_per_avg_red"])

# A pipeline can ask each step for output names.
pipeline = Pipeline(
    [
        ("add_ratio", BrightnessRatioAdder()),
        ("scale", StandardScaler()),
    ]
)

transformed = pipeline.fit_transform(image_features)
feature_names = pipeline[:-1].get_feature_names_out()
preview = pd.DataFrame(transformed, columns=feature_names).round(2)

print("scikit-learn version:", sklearn.__version__)
print("Original feature names:", list(image_features.columns))
print("Transformed feature names:", list(feature_names))
print("Transformed shape:", transformed.shape)
print(preview.head(3).to_string(index=False))



# <font color="#418FDE" size="6.5" uppercase>**Images Custom**</font>


In this lecture, you learned to:
- Represent images as arrays and extract patch-based features. 
- Create graph structures for image neighborhoods and regular grids. 
- Implement simple custom transformers compatible with scikit-learn pipelines. 

In the next Module (Module 8), we will go over 'Kernels Approximation'